# Comparative Tail Risk Analysis: S&P 500 vs CSI 300

This notebook implements a comparative tail-risk analysis of the S&P 500 and CSI 300 indices using three VaR/CVaR estimation approaches and a rolling-window backtesting framework.

| | |
|---|---|
| **Coverage** | January 2010 -- December 2023 (~14 years of daily data) |
| **Confidence levels** | 95% and 99% |
| **Models** | Historical Simulation, Normal Distribution, EWMA (lambda = 0.94) |
| **Backtesting** | Traffic Light system based on exception counts |
| **Estimation window** | Rolling 4-year window (4 x 252 = 1,008 trading days) |

---

## 1. Imports and Configuration

In [ ]:
import warnings
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

warnings.filterwarnings("ignore")

# Paths relative to this notebooks/ directory
DATA_DIR = Path("../data/raw")
FIGURES_DIR = Path("../dashboard/assets/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.family": "serif",
    "figure.dpi": 120,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

## 2. Data Loading and Preprocessing

Both datasets were sourced from Investing.com and Yahoo Finance and cover daily OHLC price data.  
We use the closing price to derive simple daily returns.

In [ ]:
sp500_raw  = pd.read_excel(DATA_DIR / "sp500.xlsx")
csi300_raw = pd.read_excel(DATA_DIR / "csi300.xlsx")

# Strip leading/trailing whitespace from column names (present in raw files)
sp500_raw.columns  = sp500_raw.columns.str.strip()
csi300_raw.columns = csi300_raw.columns.str.strip()

def prepare_data(df):
    df = df.dropna().copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    df["Daily_Return"] = df["Close"].pct_change()
    return df.dropna(subset=["Daily_Return"]).reset_index(drop=True)

sp500  = prepare_data(sp500_raw)
csi300 = prepare_data(csi300_raw)

print(f"S&P 500 : {len(sp500):,} obs  "
      f"{sp500['Date'].min().date()} to {sp500['Date'].max().date()}")
print(f"CSI 300 : {len(csi300):,} obs  "
      f"{csi300['Date'].min().date()} to {csi300['Date'].max().date()}")

## 3. Return Distribution Overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, label, color in zip(
    axes,
    [sp500, csi300],
    ["S&P 500", "CSI 300"],
    ["steelblue", "darkorange"]
):
    returns = df["Daily_Return"].dropna()
    ax.hist(returns, bins=100, color=color, alpha=0.7,
            edgecolor="none", density=True)
    mu, sigma = returns.mean(), returns.std()
    x = np.linspace(returns.min(), returns.max(), 300)
    ax.plot(x, norm.pdf(x, mu, sigma), color="black", lw=1.5,
            label=f"Normal fit (mu={mu:.4f}, sigma={sigma:.4f})")
    ax.set_title(f"{label} Daily Return Distribution")
    ax.set_xlabel("Daily Return")
    ax.set_ylabel("Density")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "return_distribution.png", bbox_inches="tight")
plt.show()

stats = pd.DataFrame({
    "S&P 500": sp500["Daily_Return"].describe(),
    "CSI 300": csi300["Daily_Return"].describe()
}).round(6)
print(stats)

## 4. VaR and CVaR Estimation

### Methodology

A **rolling 4-year estimation window** (4 x 252 = 1,008 trading days) is applied. For each test-period date *t*, the preceding 1,008 observations form the estimation window.

**Three modelling approaches:**

- **Historical Simulation** -- empirical quantile of the window; no distributional assumption
- **Normal (Parametric)** -- assumes returns are normally distributed within the window
- **EWMA (lambda = 0.94)** -- exponentially weighted variance for time-varying volatility

Confidence levels: **95%** (alpha = 0.05) and **99%** (alpha = 0.01).

In [ ]:
def calculate_var_cvar_windowed(returns, window_size, confidence_level):
    """Rolling Historical, Normal, and EWMA VaR and CVaR."""
    rolling_mean = returns.rolling(window_size).mean()
    rolling_std  = returns.rolling(window_size).std()

    # --- Historical ---
    hist_var  = returns.rolling(window_size).quantile(confidence_level)
    hist_cvar = returns.rolling(window_size).apply(
        lambda x: np.mean(x[x <= np.percentile(x, confidence_level * 100)]),
        raw=True
    )

    # --- Normal (parametric) ---
    z = norm.ppf(confidence_level)
    norm_var  = pd.Series(
        norm.ppf(confidence_level, rolling_mean, rolling_std),
        index=returns.index
    )
    norm_cvar = pd.Series(
        rolling_mean + (norm.pdf(z) / (-confidence_level)) * rolling_std,
        index=returns.index
    )

    # --- EWMA (lambda = 0.94) ---
    lambda_factor = 0.94
    z_score = norm.ppf(1 - confidence_level)

    returns_arr = returns.values
    sigma2_arr  = np.full(len(returns), np.nan)
    sigma2_arr[1] = returns_arr[1] ** 2
    for i in range(2, len(returns)):
        sigma2_arr[i] = ((1 - lambda_factor) * returns_arr[i - 1] ** 2
                         + lambda_factor * sigma2_arr[i - 1])
    sigma_arr = np.sqrt(sigma2_arr)

    ewma_var_arr  = np.full(len(returns), np.nan)
    ewma_cvar_arr = np.full(len(returns), np.nan)
    for k in range(len(returns)):
        if np.isnan(sigma_arr[k]):
            continue
        threshold = -z_score * sigma_arr[k]
        ewma_var_arr[k] = threshold
        below = returns_arr[returns_arr <= threshold]
        ewma_cvar_arr[k] = np.mean(below) if len(below) > 0 else np.nan

    ewma_var  = pd.Series(ewma_var_arr,  index=returns.index)
    ewma_cvar = pd.Series(ewma_cvar_arr, index=returns.index)

    return hist_var, hist_cvar, norm_var, norm_cvar, ewma_var, ewma_cvar

## 5. Compute Rolling Risk Metrics

> **Note:** Each EWMA loop is O(n) for sigma-squared and O(n) for VaR/CVaR.  
> Expect a few minutes of runtime for all four index/confidence-level combinations.

In [ ]:
WINDOW = 4 * 252  # 1,008-day rolling estimation window

for label, df in [("S&P 500", sp500), ("CSI 300", csi300)]:
    for cl, suffix in [(0.05, "95"), (0.01, "99")]:
        print(f"  {label} @ {100*(1-cl):.0f}% CL ...", end=" ", flush=True)
        (
            df[f"Hist_VaR_{suffix}"], df[f"Hist_CVaR_{suffix}"],
            df[f"Norm_VaR_{suffix}"], df[f"Norm_CVaR_{suffix}"],
            df[f"EWMA_VaR_{suffix}"], df[f"EWMA_CVaR_{suffix}"]
        ) = calculate_var_cvar_windowed(df["Daily_Return"], WINDOW, confidence_level=cl)
        print("done")

print("All metrics computed.")

## 6. Comparative Visualisations

The test window begins after the initial 4-year estimation period (approximately 2014 onward).

In [ ]:
COLORS = {"Historical": "#7b2d8b", "Normal": "#2e7d32", "EWMA": "#c62828"}


def plot_risk_series(df, index_label, cl_label, var_cols, cvar_cols, fig_prefix):
    data = df.copy()
    cutoff = data["Date"].iloc[0] + pd.DateOffset(years=4)
    data = data[data["Date"] > cutoff]
    model_names = ["Historical", "Normal", "EWMA"]

    for metric_type, cols in [("VaR", var_cols), ("CVaR", cvar_cols)]:
        fig, ax = plt.subplots(figsize=(15, 5))
        ax.plot(data["Date"], data["Daily_Return"],
                color="#aaaaaa", lw=0.6, alpha=0.8, label="Daily Return", zorder=1)
        for col, name in zip(cols, model_names):
            ax.plot(data["Date"], data[col], lw=1.3,
                    label=name, color=COLORS[name])
        ax.set_title(f"{index_label} -- {metric_type} at {cl_label} Confidence Level")
        ax.set_xlabel("Date")
        ax.set_ylabel(f"Daily Return / {metric_type}")
        ax.legend(loc="lower left", fontsize=9)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.xaxis.set_major_locator(mdates.YearLocator())
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f"{fig_prefix}_{metric_type.lower()}.png",
                    bbox_inches="tight")
        plt.show()


for index_label, df, prefix in [
    ("S&P 500", sp500,  "sp500"),
    ("CSI 300", csi300, "csi300")
]:
    for cl_label, suffix in [("95%", "95"), ("99%", "99")]:
        plot_risk_series(
            df, index_label, cl_label,
            var_cols=[f"Hist_VaR_{suffix}", f"Norm_VaR_{suffix}", f"EWMA_VaR_{suffix}"],
            cvar_cols=[f"Hist_CVaR_{suffix}", f"Norm_CVaR_{suffix}", f"EWMA_CVaR_{suffix}"],
            fig_prefix=f"{prefix}_{suffix}"
        )

## 7. Backtesting: Traffic Light System

The Traffic Light Test counts **exceptions** -- days where the realised return fell below the estimated VaR/CVaR -- and compares the count to the expected number over the test period.

**Expected exceptions** (test period approx. 10 years x 242 trading days/year):
- At 95% CL: 242 x 10 x 0.05 = **121**
- At 99% CL: 242 x 10 x 0.01 = **24.2**

**Classification** (deviation = actual minus expected):
- Green  -- deviation >= -5% of expected
- Yellow -- -15% <= deviation < -5%
- Red    -- deviation < -15% of expected

In [ ]:
def count_exceptions(df, return_col, risk_col):
    valid = df[[return_col, risk_col]].dropna()
    return int((valid[return_col] < valid[risk_col]).sum())


def traffic_light(actual, expected):
    dev = actual - expected
    if dev >= -0.05 * expected:
        return "Green"
    elif dev >= -0.15 * expected:
        return "Yellow"
    return "Red"


def build_backtest_table(sp_df, csi_df, expected, suffix):
    rows = []
    for name, col in [
        ("Hist VaR",  f"Hist_VaR_{suffix}"),
        ("Hist CVaR", f"Hist_CVaR_{suffix}"),
        ("Norm VaR",  f"Norm_VaR_{suffix}"),
        ("Norm CVaR", f"Norm_CVaR_{suffix}"),
        ("EWMA VaR",  f"EWMA_VaR_{suffix}"),
        ("EWMA CVaR", f"EWMA_CVaR_{suffix}"),
    ]:
        sp_exc  = count_exceptions(sp_df,  "Daily_Return", col)
        csi_exc = count_exceptions(csi_df, "Daily_Return", col)
        rows.append({
            "Metric":     name,
            "Expected":   round(expected, 1),
            "SP500 Exc.": sp_exc,
            "SP500 TL":   traffic_light(sp_exc,  expected),
            "CSI Exc.":   csi_exc,
            "CSI TL":     traffic_light(csi_exc, expected),
        })
    return pd.DataFrame(rows)


tl_95 = build_backtest_table(sp500, csi300, expected=242 * 10 * 0.05, suffix="95")
tl_99 = build_backtest_table(sp500, csi300, expected=242 * 10 * 0.01, suffix="99")

print("Traffic Light -- 95% Confidence Level")
print(tl_95.to_string(index=False))
print()
print("Traffic Light -- 99% Confidence Level")
print(tl_99.to_string(index=False))

### Traffic Light Visualisation

In [ ]:
from matplotlib.patches import Patch

TL_COLORS = {"Green": "#4caf50", "Yellow": "#ffc107", "Red": "#f44336"}


def plot_traffic_light(tl_df, cl_label, fig_name):
    expected = tl_df["Expected"].iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    for ax, dataset, exc_col, tl_col in zip(
        axes,
        ["S&P 500", "CSI 300"],
        ["SP500 Exc.", "CSI Exc."],
        ["SP500 TL",   "CSI TL"]
    ):
        bar_colors = [TL_COLORS[t] for t in tl_df[tl_col]]
        ax.bar(tl_df["Metric"], tl_df[exc_col], color=bar_colors, edgecolor="white")
        ax.axhline(expected, color="black", linestyle="--", lw=1.5,
                   label=f"Expected ({expected})")
        ax.set_title(f"{dataset} -- {cl_label}")
        ax.set_xlabel("Risk Measure")
        ax.set_ylabel("Exception Count")
        ax.legend(fontsize=9)
        ax.tick_params(axis="x", rotation=30)

    legend_elements = [Patch(facecolor=c, label=l)
                       for l, c in TL_COLORS.items()]
    fig.legend(handles=legend_elements, loc="upper center",
               ncol=3, bbox_to_anchor=(0.5, 1.03), fontsize=10)
    plt.suptitle(f"Traffic Light Backtesting -- {cl_label} Confidence Level",
                 fontsize=13, y=1.06)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / fig_name, bbox_inches="tight")
    plt.show()


plot_traffic_light(tl_95, "95%", "traffic_light_95.png")
plot_traffic_light(tl_99, "99%", "traffic_light_99.png")

## 8. Discussion and Limitations

**Key observations:**

- At the **95% confidence level**, both indices exhibit broadly acceptable Traffic Light performance across all three modelling approaches.
- At the **99% confidence level**, differences between markets and models become more pronounced. The Normal VaR model shows divergent behaviour across the two indices, reflecting how well the normality assumption fits each market.
- **Historical Simulation** tends to produce more stable exception rates, making no distributional assumption.
- The **EWMA** approach adapts to volatility clustering, producing tighter estimates in calm periods and wider estimates after large market moves.

**Limitations:**

1. **Distributional assumptions** -- The Normal model assumes i.i.d. Gaussian returns; equity returns are typically leptokurtic with heavier tails.
2. **Sample-period sensitivity** -- Results covering 2010-2023 span both a low-volatility regime and significant stress events.
3. **Model risk** -- No single VaR/CVaR model dominates across all regimes; results should be read alongside complementary risk measures.
4. **Research scope** -- This is an academic/educational project; nothing here constitutes investment advice.

**References:**

- Cogneau, P. "Backtesting Value-at-Risk: how good is the model?" *Intelligent Risk*, PRMIA, 2015.
- Rockafellar, R.T. & Uryasev, S. "Conditional Value-at-Risk for General Loss Distributions." *Journal of Banking and Finance*, Vol. 26, 2002, pp. 1443-1471.